<a href="https://colab.research.google.com/github/YomnaHadad/License-Plate-Recognition-with-YOLOv8-and-OCR/blob/main/YOLO%2BOCR_LicensePlateRecognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **YOLO+OCR for License Plate Recognition**

In [1]:
!pip install -q roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.0/285.0 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 115.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 5.6 MB/s eta 0:00:00


In [2]:
from roboflow import Roboflow
os.getenv("ROBOFLOW_API_KEY")
rf = Roboflow(api_key="wQrEpIs3LPvcZdbAKAAn")
project = rf.workspace("roboflow-universe-projects").project("license-plate-recognition-rxg4e")
version = project.version(13)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to License-Plate-Recognition-13 in yolov8:: 100%|██████████| 203744/203744 [00:30<00:00, 6781.01it/s]


In [3]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.8 MB/s eta 0:00:00


In [4]:
from ultralytics import YOLO
import shutil, os
# shutil for managing the folder and the files

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [7]:
# loading pretrained yolov8 nano model
model = YOLO("yolov8n.pt")

# finetuning part:
results = model.train(
    data = dataset.location + "/data.yaml",
    epochs = 5,
    imgsz = 640,
    batch = 16,
    patience = 3,
    workers = 4,
    device = 0
)

os.makedirs("save_models", exist_ok=True)
shutil.copy("runs/detect/train/weights/best.pt",
            "save_models/best_license_plate_recognition.pt")
print("Weights saved in saved_models/")

Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/License-Plate-Recognition-13/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64,

Now we have the fine-tuned model

In [8]:
!pip install -q easyocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.2/296.2 kB 31.0 MB/s eta 0:00:00


In [27]:
import cv2
import numpy as np
from ultralytics import YOLO
import easyocr
import re
from collections import defaultdict, deque

model = YOLO("save_models/best_license_plate_recognition.pt")
reader = easyocr.Reader(['en'], gpu=True)

#### modifing the plate to match any plate as long as it's numbers and alpha
# since License plate formats vary significantly by region.
plate_pattern = re.compile(r"^[A-Z0-9]+$")

**Mapping to closed num or alpha**

In [ ]:
# def correct_plate_format(ocr_text):
#   mapping_num_to_alpha = { "0": "O", "1": "I", "2": "Z", "5": "S", "8": "B"}
#   mapping_alpha_to_num = {"O": "0", "I": "1", "Z": "2", "S": "5", "B":"8"}

#   ocr_text = ocr_text.upper().replace(" ", "")
#   if len(ocr_text):
#     # return ""

#     corrected = []
#     for i, ch in enumerate(ocr_text):
#       # wrong number to correct alpha
#       if i < 2 or i >= 4:   # alphabet positions
#         if ch.isdigit() and ch in mapping_num_to_alpha :
#           corrected.append(mapping_num_to_alpha[ch])
#         elif ch.isalpha():
#           corrected.append(ch)
#         else:
#           return ""

#       else:
#         # wrong alpha to correct number
#         if ch.isalpha() and ch in mapping_alpha_to_num:
#           corrected.append(mapping_alpha_to_num[ch])
#         elif ch.isdigit():
#           corrected.append(ch)
#         else:
#           return ""
#     return "".join(corrected)



In [44]:
def correct_plate_format(ocr_text):
  ocr_text = ocr_text.upper().replace(" ", "")
  return ocr_text

**precrocessing the license plate region before OCR**


In [45]:
def recognize_plate(plate_crop):
  '''precrocessing the license plate region before OCR'''
    if plate_crop.size == 0:
        return ""

    gray = cv2.cvtColor(plate_crop, cv2.COLOR_BGR2GRAY)
    # _ , thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    plate_resized = cv2.resize(gray, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)

    ocr_result = reader.readtext(
    plate_resized,
    detail=0,
    allowlist="ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
    )
    if len(ocr_result) > 0:
        candidate = correct_plate_format(ocr_result[0])
        if candidate and plate_pattern.match(candidate):
          return candidate
    return ""


**Number Plate Stabilization to reduce flickering**

In [46]:
plate_history = defaultdict(lambda: deque(maxlen=10))   # last 10 prediction per boxes

plate_final = {}
def get_box_id(x1, y1, x2, y2):
  # use rounded coordinates as a pseudo id
    return f"{int(x1/10)}_{int(y1/10)}_{int(x2/10)}_{int(y2/10)}"

def get_stable_plate(box_id, new_text):

    if new_text:
        plate_history[box_id].append(new_text)

        most_common = max(
            set(plate_history[box_id]),
            key=plate_history[box_id].count
        )

        plate_final[box_id] = most_common

    return plate_final.get(box_id, "")

**Video for infernece**

In [58]:
input_video = "test_ocr_yolo101.mp4"
output_video = "output_with_license_video.mp4"
cap = cv2.VideoCapture(input_video)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(
    output_video, fourcc,
    cap.get(cv2.CAP_PROP_FPS),
    (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
     int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
)
Conf_thresh = 0.3


In [59]:
frame_count = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    # Perform YOLO inference
    results = model.predict(frame, conf=Conf_thresh)

    for result in results:
        for *xyxy, conf, cls in result.boxes.data:
            x1, y1, x2, y2 = map(int, xyxy)

            # Crop license plate region
            plate_crop = frame[y1:y2, x1:x2]

            # Recognize plate using OCR
            recognized_text = recognize_plate(plate_crop)

            # Get stable plate text
            box_id = get_box_id(x1, y1, x2, y2)
            stable_plate_text = get_stable_plate(box_id, recognized_text)

            # Draw bounding box and text on the frame
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2) # Green box
            if stable_plate_text:
                cv2.putText(frame, stable_plate_text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

    out.write(frame)

cap.release()
out.release()
print(f"Processed {frame_count} frames. Output video saved to {output_video}")


0: 384x640 3 License_Plates, 11.5ms
Speed: 4.0ms preprocess, 11.5ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 License_Plates, 12.7ms
Speed: 3.6ms preprocess, 12.7ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 License_Plates, 10.1ms
Speed: 3.6ms preprocess, 10.1ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 License_Plates, 11.5ms
Speed: 3.8ms preprocess, 11.5ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 License_Plates, 15.9ms
Speed: 5.5ms preprocess, 15.9ms inference, 2.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 License_Plates, 12.2ms
Speed: 3.4ms preprocess, 12.2ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 License_Plates, 11.1ms
Speed: 3.7ms preprocess, 11.1ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 License_Plates, 11.3ms
Speed: 3.9ms preprocess, 

In [57]:
import gradio as gr

def process_image_for_plates(img):
    # Make a copy of the image to draw on
    annotated_image = img.copy()
    detected_plates = []

    # Perform YOLO inference
    results = model.predict(img, conf=Conf_thresh)

    for result in results:
        for *xyxy, conf, cls in result.boxes.data:
            x1, y1, x2, y2 = map(int, xyxy)

            # Crop license plate region, ensuring coordinates are within bounds
            # And convert to numpy array if it's a torch tensor or similar
            plate_crop = annotated_image[max(0, y1):min(img.shape[0], y2), max(0, x1):min(img.shape[1], x2)]

            if plate_crop.shape[0] > 0 and plate_crop.shape[1] > 0:
                # Recognize plate using OCR
                recognized_text = recognize_plate(plate_crop)

                if recognized_text:
                    detected_plates.append(recognized_text)

                    # Draw bounding box and text on the annotated image
                    cv2.rectangle(annotated_image, (x1, y1), (x2, y2), (0, 255, 0), 2) # Green box
                    cv2.putText(annotated_image, recognized_text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

    # If no plates are detected, return an empty string for the plate texts
    return annotated_image, ", ".join(detected_plates) if detected_plates else "No plates detected."


# Create Gradio Interface
iface = gr.Interface(
    fn=process_image_for_plates,
    inputs=gr.Image(type="numpy", label="Upload Image"),
    outputs=[
        gr.Image(type="numpy", label="Annotated Image"),
        gr.Textbox(label="Detected Plate Numbers")
    ],
    title="License Plate Recognition with YOLOv8 and OCR",
    description="Upload an image to detect and recognize license plates."
)

# Launch the Gradio app
iface.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://74b1cc1e9bf40621a7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



0: 448x640 1 License_Plate, 45.4ms
Speed: 3.1ms preprocess, 45.4ms inference, 1.9ms postprocess per image at shape (1, 3, 448, 640)
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://74b1cc1e9bf40621a7.gradio.live


**Done:)**